# imports

In [7]:
from preprocess import get_data
from model_utils import export_model_data_json, get_auc_roc
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_validate, train_test_split

In [8]:
new_df = get_data()
new_df

,VITESSE_MA,METEO_beau,METEO_chute de neige,METEO_couvert,METEO_grêle,METEO_pluie,METEO_pluie givrante,ETAT_ROUTE_humide,ETAT_ROUTE_mouillé,ETAT_ROUTE_route enneigée,ETAT_ROUTE_sèche,ETAT_ROUTE_verglacée,LUMIERE_inconnu,LUMIERE_jour,LUMIERE_nuit,HEURE_SIN,HEURE_COS,VICTIME
0,50.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-0.866,-0.500,False
1,80.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.000,-1.000,False
2,50.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-0.707,0.707,False
3,50.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.000,0.000,False
4,80.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.866,-0.500,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18586,50.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.259,-0.966,False
18587,120.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.707,0.707,False
18588,50.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.259,-0.966,False
18589,50.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.500,-0.866,False


In [9]:
X_train, X_temp, y_train, y_temp = train_test_split(
    new_df.drop(columns=["VICTIME"]),
    new_df[["VICTIME"]],
    test_size=0.2,
    random_state=42,
)

In [10]:
def print_cross_validation_results(regressor, X_train, y_train):
    scoring = ["accuracy", "f1", "roc_auc"]    
    cv_results = cross_validate(
        regressor,
        X_train,
        y_train,
        cv=5,
        scoring=scoring,
        n_jobs=-1,
        verbose=0,
        return_train_score=True,
    )
    print("Train Accuracy: {:.2f}".format(cv_results["train_accuracy"].mean()))
    print("Validation Accuracy: {:.2f}".format(cv_results["test_accuracy"].mean()))
    print("Train F1-Score: {:.2f}".format(cv_results["train_f1"].mean()))
    print("Validation F1-Score: {:.2f}".format(cv_results["test_f1"].mean()))
    print("Train AUC-ROC: {:.2f}".format(cv_results["train_roc_auc"].mean()))
    print("Validation AUC-ROC: {:.2f}".format(cv_results["test_roc_auc"].mean()))

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    new_df.drop(columns=["VICTIME"]),
    new_df[["VICTIME"]],
    test_size=0.4,
    random_state=42,
)

dummy_clf = DummyClassifier(strategy="most_frequent", random_state=42)

dummy_clf.fit(X_train, y_train)

print_cross_validation_results(dummy_clf, X_train, y_train)

Train Accuracy: 0.58
Validation Accuracy: 0.58
Train F1-Score: 0.74
Validation F1-Score: 0.74
Train AUC-ROC: 0.50
Validation AUC-ROC: 0.50


In [14]:
y_proba_test = dummy_clf.predict_proba(X_test)[:, 1]

auc, roc = get_auc_roc(y_test, y_proba_test)

export_model_data_json("random", auc, roc)